<a href="https://colab.research.google.com/github/greek-nlp/benchmark/blob/main/gec_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Greek NLP Benchmark Suite in Colab

This notebook is the Colab entrypoint for the benchmark tasks in this repository.

Current state:
- The notebook sets up Ollama inside Colab.
- It organizes the full task suite in one place.
- It runs the reusable GEC benchmark end to end.
- It points the remaining tasks to the task-specific notebooks already present in the repo.


## Quick Start

Run the notebook top to bottom:
1. Setup cells
2. Suite configuration
3. Model pull cell
4. Task overview
5. GEC benchmark cells

Important: in Ollama, use `llama3.1:8b`, not `llama3.1:8b-instruct`.


## Repo Setup

Open this notebook from the repository in Colab, or clone/upload the repository files into the runtime before continuing.


## 1. Environment Setup

In [ ]:
!apt-get -qq update
!apt-get -qq install -y zstd

In [ ]:
!pip -q install pandas pywer openpyxl

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print("Started Ollama server with PID", ollama_process.pid)

## 2. Suite Configuration

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
print("Working directory:", project_root)
print("Python executable:", sys.executable)

MODEL_PRESETS = {
    "t4": [
        "qwen2.5:3b",
        "gemma2:2b",
        "mistral:7b",
        "qwen2.5:7b-instruct",
    ],
    "high_memory": [
        "llama3.1:8b",
        "aya-expanse:8b",
        "gemma2:9b",
        "qwen2.5:14b",
    ],
}

TASKS = {
    "toxicity": {
        "datasets": ["OGTD / OffenseEval Greek"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
    "gec": {
        "datasets": ["GNC / Korre"],
        "entrypoints": ["gec_benchmark_colab.ipynb", "gec_benchmark_notebook.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "python_module",
    },
    "machine_translation": {
        "datasets": ["PGV"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
    "summarization": {
        "datasets": ["GreekLegalSum"],
        "entrypoints": ["nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
    "intent_classification": {
        "datasets": ["UniWay GR"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
    "ner": {
        "datasets": ["elNER", "ner_nel_greek_dataset", "UniWay GR"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_access_data.ipynb"],
        "runner": "notebook",
    },
    "pos_tagging": {
        "datasets": ["UD Greek GDT"],
        "entrypoints": ["nlp_gr_access_data.ipynb"],
        "runner": "notebook",
    },
    "clustering": {
        "datasets": ["Greek Legal Code"],
        "entrypoints": ["nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
    "authorship_attribution": {
        "datasets": ["author datasets in repo notebooks"],
        "entrypoints": ["gr_attribution_zeroshot.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
}

MODEL_PRESET = "t4"
MODELS = MODEL_PRESETS[MODEL_PRESET]
ACTIVE_TASKS = list(TASKS)

SAMPLE_SIZE = 10
RANDOM_STATE = 42
OUTPUT_ROOT = Path("results/colab_suite")

print("Model preset:", MODEL_PRESET)
print("Models:", MODELS)
print("Tasks:", ACTIVE_TASKS)

In [ ]:
for model in MODELS:
    print(f"Pulling {model}...")
    subprocess.run(["ollama", "pull", model], check=True)

In [ ]:
!ollama list

## 3. Task Overview

In [ ]:
task_rows = []
for task_name, info in TASKS.items():
    task_rows.append(
        {
            "task": task_name,
            "runner": info["runner"],
            "datasets": ", ".join(info["datasets"]),
            "entrypoints": ", ".join(info["entrypoints"]),
        }
    )

pd.DataFrame(task_rows)

## 4. Notebook Routing for Non-GEC Tasks

Only GEC currently has a reusable Python benchmark module in this repository.

For the remaining tasks, start from these notebooks:
- Toxicity: `air_gr_nlp_benchmark.ipynb`, `nlp_gr_experiments.ipynb`
- Machine translation: `air_gr_nlp_benchmark.ipynb`, `nlp_gr_experiments.ipynb`
- Summarization: `nlp_gr_experiments.ipynb`
- Intent classification: `air_gr_nlp_benchmark.ipynb`, `nlp_gr_experiments.ipynb`
- NER: `air_gr_nlp_benchmark.ipynb`, `nlp_gr_access_data.ipynb`
- POS tagging: `nlp_gr_access_data.ipynb`
- Clustering: `nlp_gr_experiments.ipynb`
- Authorship attribution: `gr_attribution_zeroshot.ipynb`, `nlp_gr_experiments.ipynb`


## 5. Runnable Benchmark: GEC

In [ ]:
from gec_benchmark import benchmark_ollama_models, load_gec_dataset, save_results

GEC_MODELS = MODELS
GEC_OUTPUT_DIR = OUTPUT_ROOT / "gec"

print("Running GEC with:", GEC_MODELS)
print("Output directory:", GEC_OUTPUT_DIR)

In [ ]:
dataset = load_gec_dataset()
print("Dataset size:", len(dataset))
dataset.head()

In [ ]:
summary, raw = benchmark_ollama_models(
    models=GEC_MODELS,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

summary

In [ ]:
save_results(summary, raw, GEC_OUTPUT_DIR)
print(f"Saved benchmark outputs to {GEC_OUTPUT_DIR}")
summary